# Graph-to-MLP Knowledge Distillation (GLNN)

Knowledge Distillation on Cora (Planetoid): Distilling relational knowledge from teacher GNNs into inference-efficient student MLPs. This notebook implements the approach with `GCN / MLP` inside a `TeacherGCN` model, trained with the Adam optimizer for 60 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GCN / MLP` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Planetoid
from k3_node import transforms as k3_transforms

title = "Graph-to-MLP Knowledge Distillation (GLNN) on Cora"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora", transform=k3_transforms.NormalizeFeatures())
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

# 2. Teacher Model: GCN
class TeacherGCN(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.GCNConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.GCNConv(hidden_channels, out_channels)
        self.dropout = layers.Dropout(0.5)

    def call(self, inputs, edge_index=None, training=False):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = self.dropout(x, training=training)
        x = ops.relu(self.conv1(x, edge_index))
        x = self.dropout(x, training=training)
        return self.conv2(x, edge_index)

# 3. Student Model: Pure MLP (no graph edges at test time!)
class StudentMLP(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.lin1 = layers.Dense(hidden_channels, activation="relu")
        self.dropout = layers.Dropout(0.5)
        self.lin2 = layers.Dense(out_channels)

    def call(self, x, training=False):
        x = self.lin1(x)
        x = self.dropout(x, training=training)
        return self.lin2(x)

teacher = TeacherGCN(num_features, 64, num_classes)
student = StudentMLP(num_features, 64, num_classes)

# 4. Train Teacher
teacher.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01, weight_decay=5e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

def teacher_gen():
    mask = ops.cast(data.train_mask, "float32")
    while True:
        yield (data.x, data.edge_index), data.y, mask

print("Training Teacher GCN...")
teacher.fit(teacher_gen(), steps_per_epoch=1, epochs=20, verbose=0)
teacher_logits = teacher((data.x, data.edge_index))
# Detach from the teacher's autograd graph: these are frozen distillation
# targets, not a path the student's optimizer should (or even can, once the
# teacher's graph is freed) backpropagate through.
teacher_soft = ops.stop_gradient(ops.softmax(teacher_logits / 1.0, axis=-1))

# 5. Train Student with Knowledge Distillation
student_opt = keras.optimizers.Adam(learning_rate=0.01)

@keras.saving.register_keras_serializable()
def distillation_loss(y_true, y_pred):
    return keras.losses.categorical_crossentropy(teacher_soft, y_pred, from_logits=True)

student.compile(
    optimizer=student_opt,
    loss=distillation_loss,
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

print("Distilling Teacher knowledge to Student MLP...")
student.fit(data.x, data.y, batch_size=len(data.x), epochs=20, verbose=0)

# 6. Evaluate Student
student_out = student(data.x)
student_pred = ops.argmax(student_out, axis=-1)
test_acc = float(ops.mean(ops.cast(ops.cast(student_pred[data.test_mask], "int64") == ops.cast(data.y[data.test_mask], "int64"), "float32")))
print(f"Distilled Student MLP Test Accuracy (without graph at test time): {test_acc:.4f}")

print("\n✓ K3-Node GLNN execution completed successfully!")
